# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR² mass spectrometry protein dataset using the `mlcroissant` library. We will load the Croissant dataset, inspect its schema, extract relevant record sets and fields, process the data, and visualize key features.

### Dataset Source
The dataset source is described by the following Croissant schema URL (click to view raw schema):

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset Croissant metadata
dataset = mlc.Dataset(croissant_url)

# Print key metadata
metadata = dataset.metadata  # This is an object, not a dictionary
print(f"Dataset: {metadata.name}")
print(f"Identifier: {getattr(metadata, 'identifier', '')}")
print(f"Description: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', '')}")
print(f"Cite As: {getattr(metadata, 'citeAs', '')}")

## 2. Data Overview

List all available record sets and their fields by `@id` as defined in the Croissant schema. This helps us discover the structure of the dataset and what can be loaded for analysis.

In [ ]:
# Get all record sets (by @id) defined in the dataset
record_sets = dataset.record_sets()
print(f"Available record sets in the dataset (referenced by @id):\n")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else 'N/A'}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else 'N/A'}")
    print(f"  Fields / Columns:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}  (name: {getattr(field, 'name', 'N/A')}, dataType: {getattr(field, 'dataType', 'N/A')})")
    print()

## 3. Data Extraction
Load data from all record sets into individual DataFrames for analysis. All references are given using the entity `@id`. (See the previous output for actual values.)

In [ ]:
# Extract record set @ids for subsequent extraction
recset_ids = [rs.id for rs in dataset.record_sets()]
print("RecordSet @ids to be loaded:", recset_ids)

# Store each extracted record set as a DataFrame
dataframes = {}
for recset_id in recset_ids:
    records = list(dataset.records(record_set=recset_id))
    dataframes[recset_id] = pd.DataFrame(records)
    print(f"RecordSet '{recset_id}' loaded: {dataframes[recset_id].shape[0]} rows, columns:")
    print(dataframes[recset_id].columns.tolist())
    print()

## 4. Exploratory Data Analysis (EDA)
We will perform basic EDA on a key record set: filtering by protein abundance, normalizing numeric columns, and grouping by a categorical field. **All fields/columns are referenced by their Croissant `@id`.**

In [ ]:
# Select the main protein table record set by its @id (update if necessary)
main_recordset_id = recset_ids[0] if recset_ids else None
assert main_recordset_id is not None, "No record sets found"
df = dataframes[main_recordset_id]
print(f"Preview of main DataFrame (RecordSet @id: {main_recordset_id}):")
display(df.head())

# List columns and attempt EDA on most relevant numeric fields (by id, as required)
print("All columns / field @ids:", list(df.columns))

# Pick a numeric field (e.g. abundance, or peptide count) for analysis
# Find common likely numeric fields
import re
num_field_candidates = [col for col in df.columns if re.search(r'count|abundance|coverage|MW|norm|pi|weight', col, re.I)]
print("Numeric field candidates:", num_field_candidates)
# Use first guess as numeric_field_id for demo (you may edit for real use)
numeric_field_id = num_field_candidates[0] if num_field_candidates else df.columns[0]

# Simple filtering
threshold = df[numeric_field_id].astype(float).mean()  # Use mean as demonstration threshold
filtered_df = df[df[numeric_field_id].astype(float) > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) / filtered_df[numeric_field_id].astype(float).std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field (e.g. protein name, accession, etc; pick likely candidate by id)
cat_field_candidates = [col for col in df.columns if re.search(r'name|type|accession|desc|category|group|label', col, re.I) and col!=numeric_field_id]
group_field_id = cat_field_candidates[0] if cat_field_candidates else None
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id}, showing mean {numeric_field_id}:")
    display(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize data distributions and/or relationships between fields from the main protein record set. You may adjust field @id selections per your dataset's schema.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the selected numeric field
plt.figure(figsize=(8,4))
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
df[numeric_field_id].hist(bins=30)
plt.title(f"Distribution of field: {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.grid(True)
plt.show()

# Boxplot by group field if available
if group_field_id:
    filtered_df.boxplot(column=numeric_field_id, by=group_field_id, rot=90, figsize=(10,5))
    plt.title(f"Boxplot of {numeric_field_id} grouped by {group_field_id}")
    plt.suptitle("")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

In this notebook, we leveraged `mlcroissant` to explore the FAIR² protein mass spectrometry dataset. After loading the schema, we identified available record sets and fields by their Croissant `@id`s, extracted data into DataFrames, applied filtering and normalization operations, and visualized key fields. This approach enables robust, schema-driven, and reproducible analysis for research use cases involving structured scientific data.

*Next steps could include deeper biological data mining, feature engineering, or training machine-learning models on the discovered features.*